In [2]:
#Import Libraries
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import folium

from tqdm import tqdm
from pathlib import Path


In [3]:
#Demo Trip DF
trip_demo = pd.DataFrame({
    "ride_id": list(range(1, 21)),

    "start_station": [
        "Station A", "Station A", "Station A", "Station A", "Station A",
        "Station B", "Station B", "Station B", "Station B",
        "Station C", "Station C", "Station C",
        "Station D", "Station D", "Station D",
        "Station E", "Station E",
        "Station A", "Station B", "Station C"
    ],

    "end_station": [
        "Station B", "Station B", "Station B", "Station C", "Station C",
        "Station A", "Station A", "Station C", "Station D",
        "Station A", "Station B", "Station E",
        "Station A", "Station C", "Station E",
        "Station A", "Station D",
        "Station E", "Station E", "Station D"
    ],

    "started_at": pd.to_datetime([
        "2025-01-01 08:00", "2025-01-01 08:15", "2025-01-01 08:30",
        "2025-01-01 09:00", "2025-01-01 09:20",
        "2025-01-01 10:00", "2025-01-01 10:15", "2025-01-01 10:40",
        "2025-01-01 11:00",
        "2025-01-01 11:30", "2025-01-01 12:00", "2025-01-01 12:20",
        "2025-01-01 13:00", "2025-01-01 13:30", "2025-01-01 14:00",
        "2025-01-01 14:30", "2025-01-01 15:00",
        "2025-01-01 15:30", "2025-01-01 16:00", "2025-01-01 16:30"
    ]),

    "ended_at": pd.to_datetime([
        "2025-01-01 08:10", "2025-01-01 08:28", "2025-01-01 08:42",
        "2025-01-01 09:18", "2025-01-01 09:35",
        "2025-01-01 10:12", "2025-01-01 10:30", "2025-01-01 10:55",
        "2025-01-01 11:18",
        "2025-01-01 11:48", "2025-01-01 12:14", "2025-01-01 12:45",
        "2025-01-01 13:20", "2025-01-01 13:48", "2025-01-01 14:20",
        "2025-01-01 14:55", "2025-01-01 15:22",
        "2025-01-01 15:58", "2025-01-01 16:25", "2025-01-01 16:50"
    ]),

    "member_casual": [
        "member", "member", "casual", "member", "casual",
        "member", "member", "casual", "member",
        "casual", "member", "casual",
        "member", "casual", "member",
        "casual", "member",
        "member", "casual", "member"
    ]
})

trip_demo

,ride_id,start_station,end_station,started_at,ended_at,member_casual
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual
5,6,Station B,Station A,2025-01-01 10:00:00,2025-01-01 10:12:00,member
6,7,Station B,Station A,2025-01-01 10:15:00,2025-01-01 10:30:00,member
7,8,Station B,Station C,2025-01-01 10:40:00,2025-01-01 10:55:00,casual
8,9,Station B,Station D,2025-01-01 11:00:00,2025-01-01 11:18:00,member
9,10,Station C,Station A,2025-01-01 11:30:00,2025-01-01 11:48:00,casual


In [6]:
#Add Station Coordinates
station_coordinates = pd.DataFrame({
    "station": [
        "Station A",
        "Station B",
        "Station C",
        "Station D",
        "Station E"
    ],
    "lat": [
        40.735,
        40.751,
        40.742,
        40.728,
        40.760
    ],
    "lng": [
        -73.991,
        -73.977,
        -73.985,
        -73.970,
        -73.995
    ]
})

station_coordinates

,station,lat,lng
0,Station A,40.735,-73.991
1,Station B,40.751,-73.977
2,Station C,40.742,-73.985
3,Station D,40.728,-73.970
4,Station E,40.760,-73.995


In [7]:
#merge
trip_demo = (
    trip_demo
    .merge(
        station_coordinates,
        left_on="start_station",
        right_on="station",
        how="left"
    )
    .rename(columns={
        "lat": "start_lat",
        "lng": "start_lng"
    })
    .drop(columns="station")
    .merge(
        station_coordinates,
        left_on="end_station",
        right_on="station",
        how="left"
    )
    .rename(columns={
        "lat": "end_lat",
        "lng": "end_lng"
    })
    .drop(columns="station")
)

trip_demo.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,start_lat,start_lng,end_lat,end_lng
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,40.735,-73.991,40.751,-73.977
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,40.735,-73.991,40.751,-73.977
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,40.735,-73.991,40.751,-73.977
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,40.735,-73.991,40.742,-73.985
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,40.735,-73.991,40.742,-73.985


In [8]:
#Map Center
map_center = [
    pd.concat([trip_demo["start_lat"], trip_demo["end_lat"]]).mean(),
    pd.concat([trip_demo["start_lng"], trip_demo["end_lng"]]).mean()
]

map_center

[np.float64(40.7427), np.float64(-73.9841)]

In [9]:
#Adding Duration
trip_demo["duration_min"] = (
    trip_demo["ended_at"] - trip_demo["started_at"]
).dt.total_seconds() / 60

trip_demo[[
    "ride_id",
    "start_station",
    "end_station",
    "started_at",
    "ended_at",
    "duration_min"
]].head()

,ride_id,start_station,end_station,started_at,ended_at,duration_min
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,10.0
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,13.0
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,12.0
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,18.0
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,15.0


In [10]:
#Adding Time Features
trip_demo["date"] = trip_demo["started_at"].dt.date
trip_demo["hour"] = trip_demo["started_at"].dt.hour
trip_demo["day_name"] = trip_demo["started_at"].dt.day_name()
trip_demo["month_name"] = trip_demo["started_at"].dt.month_name()

trip_demo.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,start_lat,start_lng,end_lat,end_lng,duration_min,date,hour,day_name,month_name
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,40.735,-73.991,40.751,-73.977,10.0,2025-01-01,8,Wednesday,January
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,40.735,-73.991,40.751,-73.977,13.0,2025-01-01,8,Wednesday,January
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,40.735,-73.991,40.751,-73.977,12.0,2025-01-01,8,Wednesday,January
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,40.735,-73.991,40.742,-73.985,18.0,2025-01-01,9,Wednesday,January
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,40.735,-73.991,40.742,-73.985,15.0,2025-01-01,9,Wednesday,January


In [16]:
#Visualizing Points with Folium
m = folium.Map(
    location=[40.74, -73.99],
    zoom_start=13
)

for _, row in station_coordinates.iterrows():
    folium.Marker(
        location=[row["lat"], row["lng"]],
        popup=row["station"]
    ).add_to(m)

m

In [17]:
#Visualizing One Trip as a Line
m = folium.Map(
    location=[40.74, -73.99],
    zoom_start=13
)

start_point = [trip_demo.loc[0, "start_lat"], trip_demo.loc[0, "start_lng"]]
end_point = [trip_demo.loc[0, "end_lat"], trip_demo.loc[0, "end_lng"]]

folium.Marker(start_point, popup="Start").add_to(m)
folium.Marker(end_point, popup="End").add_to(m)

folium.PolyLine(
    locations=[start_point, end_point],
    weight=5,
    opacity=0.8
).add_to(m)

m

In [18]:
#Flow Data
flow_data = (
    trip_demo
    .groupby(
        ["start_station", "end_station"],
        as_index=False
    )
    .agg(
        number_of_rides=("ride_id", "count"),
        avg_duration_min=("duration_min", "mean"),
        start_lat=("start_lat", "mean"),
        start_lng=("start_lng", "mean"),
        end_lat=("end_lat", "mean"),
        end_lng=("end_lng", "mean")
    )
    .sort_values("number_of_rides", ascending=False)
)

flow_data

,start_station,end_station,number_of_rides,avg_duration_min,start_lat,start_lng,end_lat,end_lng
0,Station A,Station B,3,11.666667,40.735,-73.991,40.751,-73.977
1,Station A,Station C,2,16.500000,40.735,-73.991,40.742,-73.985
3,Station B,Station A,2,13.500000,40.751,-73.977,40.735,-73.991
2,Station A,Station E,1,28.000000,40.735,-73.991,40.760,-73.995
4,Station B,Station C,1,15.000000,40.751,-73.977,40.742,-73.985
5,Station B,Station D,1,18.000000,40.751,-73.977,40.728,-73.970
6,Station B,Station E,1,25.000000,40.751,-73.977,40.760,-73.995
7,Station C,Station A,1,18.000000,40.742,-73.985,40.735,-73.991
8,Station C,Station B,1,14.000000,40.742,-73.985,40.751,-73.977
9,Station C,Station D,1,20.000000,40.742,-73.985,40.728,-73.970


In [19]:
#Visualizing Flows
flow_map = folium.Map(
    location=map_center,
    zoom_start=13,
    tiles="CartoDB positron"
)

for _, row in station_coordinates.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=6,
        popup=row["station"],
        tooltip=row["station"],
        fill=True
    ).add_to(flow_map)

max_rides = flow_data["number_of_rides"].max()

for _, row in flow_data.iterrows():

    start_point = [row["start_lat"], row["start_lng"]]
    end_point = [row["end_lat"], row["end_lng"]]

    line_width = 1 + 7 * row["number_of_rides"] / max_rides

    popup_text = (
        f"<b>{row['start_station']} → {row['end_station']}</b><br>"
        f"Number of rides: {row['number_of_rides']}<br>"
        f"Average duration: {row['avg_duration_min']:.1f} minutes"
    )

    folium.PolyLine(
        locations=[start_point, end_point],
        weight=line_width,
        opacity=0.7,
        popup=popup_text,
        tooltip=f"{row['start_station']} → {row['end_station']}"
    ).add_to(flow_map)

flow_map

In [21]:
#Bounding Box
min_lat = min(trip_demo["start_lat"].min(), trip_demo["end_lat"].min())
max_lat = max(trip_demo["start_lat"].max(), trip_demo["end_lat"].max())

min_lng = min(trip_demo["start_lng"].min(), trip_demo["end_lng"].min())
max_lng = max(trip_demo["start_lng"].max(), trip_demo["end_lng"].max())

bounding_box = pd.DataFrame({
    "metric": ["min_lat", "max_lat", "min_lng", "max_lng"],
    "value": [min_lat, max_lat, min_lng, max_lng]
})

bounding_box

,metric,value
0,min_lat,40.728
1,max_lat,40.760
2,min_lng,-73.995
3,max_lng,-73.970


In [22]:
bbox_map = folium.Map(
    location=map_center,
    zoom_start=13,
    tiles="CartoDB positron"
)

for _, row in station_coordinates.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=6,
        popup=row["station"],
        tooltip=row["station"],
        fill=True
    ).add_to(bbox_map)

bbox_coordinates = [
    [min_lat, min_lng],
    [min_lat, max_lng],
    [max_lat, max_lng],
    [max_lat, min_lng],
    [min_lat, min_lng]
]

folium.PolyLine(
    locations=bbox_coordinates,
    weight=4,
    opacity=0.8,
    popup="Bounding Box"
).add_to(bbox_map)

bbox_map

In [24]:
import geopandas as gpd
start_points = gpd.GeoDataFrame(
    trip_demo.copy(),
    geometry=gpd.points_from_xy(
        trip_demo["start_lng"],
        trip_demo["start_lat"]
    ),
    crs="EPSG:4326"
)

end_points = gpd.GeoDataFrame(
    trip_demo.copy(),
    geometry=gpd.points_from_xy(
        trip_demo["end_lng"],
        trip_demo["end_lat"]
    ),
    crs="EPSG:4326"
)

In [25]:
start_points[[
    "ride_id",
    "start_station",
    "end_station",
    "geometry"
]].head()

,ride_id,start_station,end_station,geometry
0,1,Station A,Station B,POINT (-73.991 40.735)
1,2,Station A,Station B,POINT (-73.991 40.735)
2,3,Station A,Station B,POINT (-73.991 40.735)
3,4,Station A,Station C,POINT (-73.991 40.735)
4,5,Station A,Station C,POINT (-73.991 40.735)


In [26]:
projected_crs = "EPSG:32618"

start_points_projected = start_points.to_crs(projected_crs)
end_points_projected = end_points.to_crs(projected_crs)

In [27]:
trip_demo["distance_m"] = start_points_projected.geometry.distance(
    end_points_projected.geometry
)

trip_demo["distance_km"] = trip_demo["distance_m"] / 1000

trip_demo[[
    "ride_id",
    "start_station",
    "end_station",
    "duration_min",
    "distance_m",
    "distance_km"
]].head()

,ride_id,start_station,end_station,duration_min,distance_m,distance_km
0,1,Station A,Station B,10.0,2133.621698,2.133622
1,2,Station A,Station B,13.0,2133.621698,2.133622
2,3,Station A,Station B,12.0,2133.621698,2.133622
3,4,Station A,Station C,18.0,927.671157,0.927671
4,5,Station A,Station C,15.0,927.671157,0.927671


In [28]:
trip_demo[["distance_m", "distance_km"]].describe()

,distance_m,distance_km
count,20.000000,20.000000
mean,2113.717746,2.113718
std,898.892579,0.898893
min,927.671157,0.927671
25%,1665.451089,1.665451
50%,2133.621698,2.133622
75%,2282.181051,2.282181
max,4132.274455,4.132274


In [29]:
route_summary = (
    trip_demo
    .groupby(
        ["start_station", "end_station"],
        as_index=False
    )
    .agg(
        number_of_rides=("ride_id", "count"),
        avg_duration_min=("duration_min", "mean"),
        median_duration_min=("duration_min", "median"),
        avg_distance_km=("distance_km", "mean"),
        median_distance_km=("distance_km", "median"),
        start_lat=("start_lat", "mean"),
        start_lng=("start_lng", "mean"),
        end_lat=("end_lat", "mean"),
        end_lng=("end_lng", "mean")
    )
    .sort_values("number_of_rides", ascending=False)
)

route_summary

,start_station,end_station,number_of_rides,avg_duration_min,median_duration_min,avg_distance_km,median_distance_km,start_lat,start_lng,end_lat,end_lng
0,Station A,Station B,3,11.666667,12.0,2.133622,2.133622,40.735,-73.991,40.751,-73.977
1,Station A,Station C,2,16.500000,16.5,0.927671,0.927671,40.735,-73.991,40.742,-73.985
3,Station B,Station A,2,13.500000,13.5,2.133622,2.133622,40.751,-73.977,40.735,-73.991
2,Station A,Station E,1,28.000000,28.0,2.795834,2.795834,40.735,-73.991,40.760,-73.995
4,Station B,Station C,1,15.000000,15.0,1.206023,1.206023,40.751,-73.977,40.742,-73.985
5,Station B,Station D,1,18.000000,18.0,2.620861,2.620861,40.751,-73.977,40.728,-73.970
6,Station B,Station E,1,25.000000,25.0,1.818594,1.818594,40.751,-73.977,40.760,-73.995
7,Station C,Station A,1,18.000000,18.0,0.927671,0.927671,40.742,-73.985,40.735,-73.991
8,Station C,Station B,1,14.000000,14.0,1.206023,1.206023,40.742,-73.985,40.751,-73.977
9,Station C,Station D,1,20.000000,20.0,2.005000,2.005000,40.742,-73.985,40.728,-73.970
